In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import (PatternFill, Font, Alignment, Border, Side)
from openpyxl.utils import get_column_letter
print("Imports OK")

Imports OK


In [2]:
# ============================================================
# CONFIG  – edit the items in this section only
# ============================================================
_XLSX_DIR    = r'C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260901_EVE-A3-155Ah_CyLT&CaLT'
_OUTPUT_FILE = r'C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260901_EVE-A3-155Ah_CyLT&CaLT\EVE-A3-155Ah_CyLT_MasterCode_v2.xlsx'

_CELLS = [
    # ── 25°C 0.25P ──────────────────────────────────────────────────────────
    {'xlsx': '25℃-0# F3-A01-05-01 04QCN00540100JE590200342_CY369_translated.xlsx',
     'temperature': 25, 'test_type': 'CyLT', 'p_rate': 0.25,
     'serial': 'F3-A01-05-01', 'file_folder': '', 'type_note': '', 'specific_notes': '', 'general_notes': ''},

    {'xlsx': '25℃ 0.25P Continued LNB567# F3-A01-05-02 04QCN00540100JE5A0200463_CY376_translated.xlsx',
     'temperature': 25, 'test_type': 'CyLT', 'p_rate': 0.25,
     'serial': 'F3-A01-05-02', 'file_folder': '', 'type_note': 'Continued', 'specific_notes': '', 'general_notes': ''},

    # ── 45°C 0.25P ──────────────────────────────────────────────────────────
    {'xlsx': '45℃-0# F3-B01-07-06 04QCN00540100JE5A0200458_CY379_translated.xlsx',
     'temperature': 45, 'test_type': 'CyLT', 'p_rate': 0.25,
     'serial': 'F3-B01-07-06', 'file_folder': '', 'type_note': '', 'specific_notes': '', 'general_notes': ''},

    {'xlsx': '45℃-0# F3-B01-07-07 04QCN00540100JE5C0200707_CY374_translated.xlsx',
     'temperature': 45, 'test_type': 'CyLT', 'p_rate': 0.25,
     'serial': 'F3-B01-07-07', 'file_folder': '', 'type_note': '', 'specific_notes': '', 'general_notes': ''},
]

# ============================================================
# Column rename: source Excel header → internal name used by _calc()
# ============================================================
_COL_RENAME = {
    'Cycle number':           'Cycle',
    'Charge capacity(Ah)':    'Charge_Capacity_Ah',
    'Discharge capacity(Ah)': 'Discharge_Capacity_Ah',
    'Charge energy(Wh)':      'Charge_Energy_Wh',
    'Discharge energy(Wh)':   'Discharge_Energy_Wh',
}

# ============================================================
# ENGINE  – do not edit below this line
# ============================================================

_DARK  = PatternFill('solid', fgColor='1F4E79')
_MED   = PatternFill('solid', fgColor='2E75B6')
_LGREY = PatternFill('solid', fgColor='F2F2F2')
_WHITE = PatternFill('solid', fgColor='FFFFFF')
_FW    = Font(name='Arial', bold=True, color='FFFFFF', size=10)
_FB    = Font(name='Arial', bold=True, size=10)
_FN    = Font(name='Arial', size=10)
_AC    = Alignment(horizontal='center', vertical='center', wrap_text=True)
_AL    = Alignment(horizontal='left',   vertical='center', wrap_text=True)
_AR    = Alignment(horizontal='right',  vertical='center')
_thin  = Side(style='thin', color='BFBFBF')
_BD    = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)

def _style(cell, fill=None, font=None, align=None, border=None):
    if fill:   cell.fill      = fill
    if font:   cell.font      = font
    if align:  cell.alignment = align
    if border: cell.border    = border

_CD = [
    ('Time (days)',                     None),
    ('Cycle',                           'Cycle'),
    ('Charge capacity (Ah)',            'Charge_Capacity_Ah'),
    ('Discharge capacity (Ah)',         'Discharge_Capacity_Ah'),
    ('Relative discharge capacity',     'Relative_discharge_capacity'),
    ('Coulombic efficiency',            'Coulombic_efficiency'),
    ('Charge energy (Wh)',              'Charge_Energy_Wh'),
    ('Discharge energy (Wh)',           'Discharge_Energy_Wh'),
    ('Relative discharge energy',       'Relative_discharge_energy'),
    ('Energy efficiency',               'Energy_efficiency'),
    ('Energy inefficiency',             'Energy_inefficiency'),
    ('Relative energy efficiency',      'Relative_energy_efficiency'),
    ('Relative energy inefficiency',    'Relative_energy_inefficiency'),
    ('dV charge (V)',                   None),
    ('dV discharge (V)',                None),
    ('dV charge cumulative (V)',        None),
    ('dV discharge cumulative (V)',     None),
    ('DCR charge (mOhm)',               None),
    ('DCR discharge (mOhm)',            None),
    ('DCR charge cumulative (mOhm)',    None),
    ('DCR discharge cumulative (mOhm)', None),
    ('Ambient temperature (C)',         None),
    ('Max charge temp (C)',             None),
    ('Max discharge temp (C)',          None),
    ('Thickness (mm)',                  None),
    ('Relative thickness',             None),
]
_HEADERS = [h for h, _ in _CD]
_SRCCOLS = [s for _, s in _CD]

def _read_xlsx(path):
    """Read source xlsx and normalise column names for _calc()."""
    df = pd.read_excel(path)
    df = df.rename(columns=_COL_RENAME)
    # Energy efficiency is not supplied directly; compute it from energy columns.
    df['Energy_Efficiency_pct'] = (
        df['Discharge_Energy_Wh'] / df['Charge_Energy_Wh'] * 100
    )
    return df

def _calc(df):
    df = df.copy()
    df['Coulombic_efficiency'] = np.where(
        df['Charge_Capacity_Ah'] > 0,
        df['Discharge_Capacity_Ah'] / df['Charge_Capacity_Ah'], np.nan)
    df['Energy_efficiency']   = df['Energy_Efficiency_pct'] / 100.0
    df['Energy_inefficiency'] = 1.0 - df['Energy_efficiency']
    _ref = df[df['Cycle'] == 1]
    if _ref.empty:
        for _c in ('Relative_discharge_capacity', 'Relative_discharge_energy',
                   'Relative_energy_efficiency', 'Relative_energy_inefficiency'):
            df[_c] = np.nan
        return df
    _rc = float(_ref['Discharge_Capacity_Ah'].iat[0])
    _rn = float(_ref['Discharge_Energy_Wh'].iat[0])
    _re = float(_ref['Energy_efficiency'].iat[0])
    _ri = float(_ref['Energy_inefficiency'].iat[0])
    def _sd(s, r): return s / r if r else np.nan
    df['Relative_discharge_capacity']  = _sd(df['Discharge_Capacity_Ah'], _rc)
    df['Relative_discharge_energy']    = _sd(df['Discharge_Energy_Wh'],   _rn)
    df['Relative_energy_efficiency']   = _sd(df['Energy_efficiency'],     _re)
    df['Relative_energy_inefficiency'] = _sd(df['Energy_inefficiency'],   _ri)
    return df

def _meta_sheet(ws, cells_cfg):
    ws.title = 'Metadata'
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'B3'
    _META_FIELDS = [
        ('Cell number',    'cell_num'),
        ('Serial number',  'serial'),
        ('Temperature',    'temperature'),
        ('Test type',      'test_type'),
        ('P-rate',         'p_rate'),
        ('File/folder',    'file_folder'),
        ('Type note',      'type_note'),
        ('Specific notes', 'specific_notes'),
        ('General notes',  'general_notes'),
    ]
    _N_FIELDS = len(_META_FIELDS)
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=1+_N_FIELDS)
    _h = ws.cell(1, 1, 'Cell Metadata')
    _style(_h, fill=_DARK, font=_FW, align=_AC)
    ws.cell(2, 1, 'Test')
    _style(ws.cell(2, 1), fill=_MED, font=_FW, align=_AC)
    for _ci, (_label, _key) in enumerate(_META_FIELDS, start=2):
        _hc = ws.cell(2, _ci, _label)
        _style(_hc, fill=_MED, font=_FW, align=_AC)
    for _ri, _c in enumerate(cells_cfg, start=3):
        _fill = _LGREY if (_ri % 2 == 1) else _WHITE
        _lc = ws.cell(_ri, 1, f'Cell {_c["cell_num"]}')
        _style(_lc, fill=_fill, font=_FB, align=_AL)
        for _ci, (_label, _key) in enumerate(_META_FIELDS, start=2):
            _val = _c.get(_key, '')
            _dc = ws.cell(_ri, _ci, _val)
            _style(_dc, fill=_fill, font=_FN, align=_AL)
    ws.column_dimensions['A'].width = 10
    for _ci in range(2, 2+_N_FIELDS):
        ws.column_dimensions[get_column_letter(_ci)].width = 20

def _cell_sheet(ws, label, df):
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'C3'
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(_CD))
    _h = ws.cell(1, 1, label)
    _style(_h, fill=_DARK, font=_FW, align=_AC)
    for _ci, _hdr in enumerate(_HEADERS, start=1):
        _hc = ws.cell(2, _ci, _hdr)
        _style(_hc, fill=_MED, font=_FW, align=_AC)
    for _ri, _row in df.reset_index(drop=True).iterrows():
        _fill = _LGREY if (_ri % 2 == 0) else _WHITE
        for _ci, _src in enumerate(_SRCCOLS, start=1):
            _val = _row[_src] if (_src and _src in df.columns) else 'NAN'
            _dc = ws.cell(_ri + 3, _ci, _val)
            _style(_dc, fill=_fill, font=_FN, align=_AR if _ci > 1 else _AL)
    ws.column_dimensions['A'].width = 12
    ws.column_dimensions['B'].width = 8
    for _ci in range(3, len(_CD)+1):
        ws.column_dimensions[get_column_letter(_ci)].width = 20
    ws.row_dimensions[1].height = 20
    ws.row_dimensions[2].height = 30

# ============================================================
# RUN
# ============================================================
_xlsx_dir = Path(_XLSX_DIR)
_wb = Workbook()
_wb.remove(_wb.active)
_cells_with_num = [{**_c, 'cell_num': _i+1} for _i, _c in enumerate(_CELLS)]
_ws_meta = _wb.create_sheet('Metadata')
_meta_sheet(_ws_meta, _cells_with_num)
print(f'Output : {_OUTPUT_FILE}')
print(f'Cells  : {len(_CELLS)}')
print('  [Metadata]  written')
for _i, _cfg in enumerate(_CELLS):
    _xlsx_path = _xlsx_dir / _cfg['xlsx']
    _df_raw    = _read_xlsx(_xlsx_path)
    _df        = _calc(_df_raw)
    _serial    = _cfg['serial']
    _n_rows    = len(_df)
    _cy2_rows  = _df[_df['Cycle'] == 2]
    _disch     = _cy2_rows['Discharge_Capacity_Ah'].iat[0] if not _cy2_rows.empty else float('nan')
    _ws        = _wb.create_sheet(f'Cell {_i+1}')
    _cell_sheet(_ws, _serial, _df)
    print(f'  [Cell {_i+1}]  {_serial}  |  {_n_rows} rows  |  cy2 disch cap {_disch:.4f} Ah')
Path(_OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)
_wb.save(_OUTPUT_FILE)
print(f'Saved: {_OUTPUT_FILE}')

Output : C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260901_EVE-A3-155Ah_CyLT&CaLT\EVE-A3-155Ah_CyLT_MasterCode_v2.xlsx
Cells  : 4
  [Metadata]  written
  [Cell 1]  F3-A01-05-01  |  1530 rows  |  cy2 disch cap 159.1910 Ah
  [Cell 2]  F3-A01-05-02  |  1530 rows  |  cy2 disch cap 159.4150 Ah
  [Cell 3]  F3-B01-07-06  |  1556 rows  |  cy2 disch cap 159.8320 Ah
  [Cell 4]  F3-B01-07-07  |  1549 rows  |  cy2 disch cap 161.3750 Ah
Saved: C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260901_EVE-A3-155Ah_CyLT&CaLT\EVE-A3-155Ah_CyLT_MasterCode_v2.xlsx
